In [1]:
# =========================================================
# IFRS 9 — Missing Data Treatments (Python equivalents)
# =========================================================
# Steps covered:
# 4) Mode imputation (categorical)
# 5) Hot-deck imputation (donor within strata)
# 6) Regression imputation (example: AGE)
# 7) Cluster-based imputation (KMeans + cluster means)
# 8) SME rule-based imputation
# 9) QA summaries
# 10) Optional: export an audit pack to Excel
#
# Notes:
# - Raw columns are preserved; imputed versions are added as *_imp.
# - For hot-deck, we pick a random donor within (employment_status, marital_status, region, stage).
# - For regression, we use a simple, auditable pipeline (OneHot + LinearRegression).
# - For cluster imputation, we compute cluster means on complete cases and merge back.
# =========================================================

import numpy as np
import pandas as pd
from pathlib import Path

# ML utilities for Regression Imputation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans

# -----------------------------
# 0) Load data (adjust path)
# -----------------------------

csv_path = Path("C:/Users/customer/Desktop/MyVideos/UDEMY LESSONS/IFRS 9 Python/DQ/with_missing_ifrs9_pit_pd.csv")  # e.g., r"C:\...\with_missing_ifrs9_pit_pd.csv"
df = pd.read_csv(csv_path)

# Common lists (align with your SAS code)
id_var   = "account_id"
date_var = "report_date"
num_vars = ["age", "credit_utilization", "internal_score"]
cat_vars = ["employment_status", "marital_status", "region"]

# Keep only columns that exist
num_vars = [c for c in num_vars if c in df.columns]
cat_vars = [c for c in cat_vars if c in df.columns]

# =========================================================
# 4) Mode Imputation (categorical)
#    - Adds *_imp columns for each categorical in cat_vars
# =========================================================
df_mode_imp = df.copy()

def mode_impute(series: pd.Series) -> pd.Series:
    """Return series with NaN replaced by most frequent non-null value."""
    if series.dropna().empty:
        return series  # nothing to impute
    mode_val = series.dropna().mode().iloc[0]
    return series.fillna(mode_val)

for v in cat_vars:
    df_mode_imp[f"{v}_imp"] = mode_impute(df_mode_imp[v])

# Quick QA for mode imputation (frequency tables)
for v in cat_vars:
    _freq = df_mode_imp[f"{v}_imp"].value_counts(dropna=False)
    print(f"[MODE QA] {v}_imp value counts:\n{_freq.head()}\n")




[MODE QA] employment_status_imp value counts:
employment_status_imp
Employed         3052
Unemployed        848
Self-Employed     659
Retired           441
Name: count, dtype: int64

[MODE QA] marital_status_imp value counts:
marital_status_imp
Married     2521
Single      1794
Divorced     467
Widowed      218
Name: count, dtype: int64

[MODE QA] region_imp value counts:
region_imp
West     1290
North    1269
East     1239
South    1202
Name: count, dtype: int64



In [2]:
# =========================================================
# HEAD OF ORIGINAL CATEGORICAL VARIABLES (before imputation)
# =========================================================

original_heads = df[cat_vars].head()

print("\n=== HEAD OF ORIGINAL (RAW) CATEGORICAL VARIABLES ===")

original_heads




=== HEAD OF ORIGINAL (RAW) CATEGORICAL VARIABLES ===


,employment_status,marital_status,region
0,Employed,Single,West
1,Employed,Married,West
2,Employed,NaN,South
3,Employed,Married,South
4,Employed,Married,West


In [3]:
# =========================================================
# Show head() of all imputed categorical variables
# =========================================================

# Build a mini-table showing the first few rows of each *_imp column
imputed_heads = df_mode_imp[[f"{v}_imp" for v in cat_vars]].head()

print("\n=== HEAD OF IMPUTED CATEGORICAL VARIABLES ===")

imputed_heads


=== HEAD OF IMPUTED CATEGORICAL VARIABLES ===


,employment_status_imp,marital_status_imp,region_imp
0,Employed,Single,West
1,Employed,Married,West
2,Employed,Married,South
3,Employed,Married,South
4,Employed,Married,West


In [4]:
# =========================================================
# Regression Imputation for AGE (NaN-safe, IFRS9-compliant)
# =========================================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# ---------------------------------------------
# 0) Load your dataset
# ---------------------------------------------
# df = pd.read_csv("with_missing_ifrs9_pit_pd.csv")   # if not already loaded

# ---------------------------------------------
# 1) Target & predictors
# ---------------------------------------------
target = "age"
predictors = [
    "internal_score",
    "credit_utilization",
    "macro_gdp_growth",
    "macro_unemployment",
    "macro_interest_rate"
]

predictors = [c for c in predictors if c in df.columns]

# ---------------------------------------------
# 2) Split rows with/without AGE
# ---------------------------------------------
df_train = df[df[target].notna()].copy()
df_apply = df[df[target].isna()].copy()

X_train_all = df_train[predictors]
y_train_all = df_train[target]

# ---------------------------------------------
# 3) Separate numeric & categorical predictors
# ---------------------------------------------
cat_cols = [c for c in predictors if df[c].dtype == "object"]
num_cols = [c for c in predictors if c not in cat_cols]

# ---------------------------------------------
# 4) Preprocessing pipelines
# ---------------------------------------------
cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))   # ✅ FIXED HERE
])

num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ]
)

# ---------------------------------------------
# 5) Final model
# ---------------------------------------------
pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", LinearRegression())
])

# ---------------------------------------------
# 6) Validation split (like SAS partition validate=0.30)
# ---------------------------------------------
X_tr, X_va, y_tr, y_va = train_test_split(
    X_train_all, y_train_all, test_size=0.30, random_state=42
)

pipe.fit(X_tr, y_tr)
y_va_pred = pipe.predict(X_va)

print(f"[REG QA] R² on validation = {r2_score(y_va, y_va_pred):.3f}")

# ---------------------------------------------
# 7) Predict for missing AGE rows
# ---------------------------------------------
df_out = df.copy()
df_out["age_pred"] = np.nan

# Pred for complete rows (optional auditing)
df_out.loc[df[target].notna(), "age_pred"] = pipe.predict(df_train[predictors])

# Pred for missing rows
if not df_apply.empty:
    df_out.loc[df[target].isna(), "age_pred"] = pipe.predict(df_apply[predictors])

# Final imputing
df_out["age_imp"] = np.where(df_out[target].isna(), df_out["age_pred"], df_out[target])

# ---------------------------------------------
# 8) Simple QA summary (PROC MEANS equivalent)
# ---------------------------------------------
print(df_out[["age", "age_imp"]].describe())


[REG QA] R² on validation = 0.003
               age      age_imp
count  4500.000000  5000.000000
mean     45.207556    45.208826
std      14.112070    13.388510
min      21.000000    21.000000
25%      33.000000    35.000000
50%      45.000000    45.153379
75%      57.000000    56.000000
max      69.000000    69.000000


In [5]:
# =========================================================
# 9) QA — Robust checks (safe even if a step was skipped)
#     Drop-in replacement for the last three QA cells
# =========================================================

import numpy as np
import pandas as pd

# ---------- Helper 1: numeric summary (PROC MEANS-like) ----------
def qa_means_like(frame: pd.DataFrame, columns: list[str], label: str):
    cols = [c for c in columns if c in frame.columns]
    if not cols:
        print(f"[QA] {label} — no numeric columns found in frame for: {columns}")
        return
    out = []
    for c in cols:
        s = frame[c]
        out.append({
            "variable": c,
            "n": int(s.shape[0]),
            "nmiss": int(s.isna().sum()),
            "mean": float(s.mean(skipna=True)) if pd.api.types.is_numeric_dtype(s) else np.nan,
            "median": float(s.median(skipna=True)) if pd.api.types.is_numeric_dtype(s) else np.nan,
            "std": float(s.std(skipna=True)) if pd.api.types.is_numeric_dtype(s) else np.nan,
            "min": float(s.min(skipna=True)) if pd.api.types.is_numeric_dtype(s) else np.nan,
            "p50": float(s.median(skipna=True)) if pd.api.types.is_numeric_dtype(s) else np.nan,
            "max": float(s.max(skipna=True)) if pd.api.types.is_numeric_dtype(s) else np.nan,
        })
    qa_df = pd.DataFrame(out)
    print(f"\n[QA] {label}")
    display(qa_df)


# ---------- Helper 2: numeric imputation summary (raw vs imp) ----------
def qa_imputation_summary_numeric(frame: pd.DataFrame, raw_col: str, imp_col: str) -> pd.DataFrame:
    assert raw_col in frame.columns, f"Missing column: {raw_col}"
    assert imp_col in frame.columns, f"Missing column: {imp_col}"
    raw = frame[raw_col]
    imp = frame[imp_col]

    n = int(frame.shape[0])
    n_miss_raw = int(raw.isna().sum())
    n_miss_imp = int(imp.isna().sum())
    n_imputed  = int(((raw.isna()) & (imp.notna())).sum())         # fixed parentheses/casting
    n_changed  = int(((raw.notna()) & (imp.notna()) & (raw != imp)).sum())

    return pd.DataFrame([{
        "series_raw": raw_col,
        "series_imp": imp_col,
        "n": n,
        "n_miss_raw": n_miss_raw,
        "n_miss_imp": n_miss_imp,
        "n_imputed": n_imputed,
        "n_overwritten_when_present": n_changed
    }])


# ---------- Helper 3: categorical imputation summary (raw vs sme/imp) ----------
def qa_imputation_summary_categorical(frame: pd.DataFrame, raw_col: str, sme_col: str) -> pd.DataFrame:
    assert raw_col in frame.columns, f"Missing column: {raw_col}"
    assert sme_col in frame.columns, f"Missing column: {sme_col}"
    raw = frame[raw_col]
    sme = frame[sme_col]

    n = int(frame.shape[0])
    n_miss_raw = int(raw.isna().sum())
    n_miss_sme = int(sme.isna().sum())
    n_imputed  = int(((raw.isna()) & (sme.notna())).sum())         # ✅ fixed TypeError
    n_changed  = int(((raw.notna()) & (sme.notna()) & (raw != sme)).sum())

    return pd.DataFrame([{
        "series_raw": raw_col,
        "series_sme": sme_col,
        "n": n,
        "n_miss_raw": n_miss_raw,
        "n_miss_sme": n_miss_sme,
        "n_imputed_from_missing": n_imputed,
        "n_overwritten_when_present": n_changed
    }])


# ---------- Helper 4: show head() side-by-side ----------
def qa_show_heads(frame: pd.DataFrame, cols: list[str], label: str, n: int = 5):
    keep = [c for c in cols if c in frame.columns]
    if not keep:
        print(f"[HEAD] {label} — columns not found: {cols}")
        return
    print(f"\n=== HEAD: {label} ===")
    display(frame[keep].head(n))


# =========================================================
# QA CELL 1 — Mode & Hot-deck
#   • Works even if df_hotdeck isn't present (skips gracefully)
# =========================================================
if 'df_mode_imp' in globals():
    print("[QA] Mode step (categoricals shown above) — OK")
else:
    print("[QA] Mode step — df_mode_imp not in environment, skipping")

if 'df_hotdeck' in globals():
    qa_means_like(df_hotdeck,
                  ["credit_utilization_raw", "credit_utilization_imp"],
                  "Hot-deck credit_utilization")
    qa_show_heads(df_hotdeck,
                  ["credit_utilization_raw", "credit_utilization_imp"],
                  "CREDIT_UTILIZATION (RAW vs IMP)")
    try:
        display(qa_imputation_summary_numeric(df_hotdeck, "credit_utilization_raw", "credit_utilization_imp"))
    except AssertionError as e:
        print("[QA] Hot-deck summary skipped:", e)
else:
    print("[QA] Hot-deck — df_hotdeck not in environment, skipping")


# =========================================================
# QA CELL 2 — Cluster-based internal_score
# =========================================================
if 'df_fastclus' in globals():
    qa_means_like(df_fastclus,
                  ["internal_score", "internal_score_imp"],
                  "Cluster-based internal_score")
    qa_show_heads(df_fastclus,
                  ["internal_score", "internal_score_imp", "cluster"],
                  "INTERNAL_SCORE (RAW vs IMP) + CLUSTER", n=10)
    try:
        display(qa_imputation_summary_numeric(df_fastclus, "internal_score", "internal_score_imp"))
    except AssertionError as e:
        print("[QA] Cluster summary skipped:", e)
else:
    print("[QA] Cluster-based — df_fastclus not in environment, skipping")


# =========================================================
# QA CELL 3 — SME rule-based imputations
#   • Shows where imputation happened and cross-tab
# =========================================================
cat_pairs = []
if 'df_sme' in globals():
    if "employment_status" in df_sme.columns and "employment_status_sme" in df_sme.columns:
        cat_pairs.append(("employment_status", "employment_status_sme"))
    if "marital_status" in df_sme.columns and "marital_status_sme" in df_sme.columns:
        cat_pairs.append(("marital_status", "marital_status_sme"))

    for raw_col, sme_col in cat_pairs:
        qa_show_heads(df_sme, [raw_col, sme_col], f"{raw_col.upper()} (RAW vs SME)", n=5)
        # top counts
        vc = df_sme[sme_col].value_counts(dropna=False).reset_index()
        vc.columns = [sme_col, "count"]
        vc["pct"] = (vc["count"] / vc["count"].sum() * 100).round(2)
        print(f"\n=== SME QA: {sme_col.upper()} — TOP COUNTS (incl. NaN) ===")
        display(vc.head(10))

        # summary table (fixed TypeError)
        print(f"\n=== SME QA: {raw_col.upper()} vs {sme_col.upper()} — SUMMARY ===")
        display(qa_imputation_summary_categorical(df_sme, raw_col, sme_col))

        # show where SME filled from missing raw
        filled = df_sme[(df_sme[raw_col].isna()) & (df_sme[sme_col].notna())].copy()
        cols_to_show = [c for c in [globals().get('id_var', None), raw_col, sme_col] if c]
        print(f"\n=== SME FILLS (RAW missing → SME provided) — sample {min(20, len(filled))} rows ===")
        display(filled[cols_to_show].head(20))
else:
    print("[QA] SME — df_sme not in environment, skipping")


[QA] Mode step (categoricals shown above) — OK
[QA] Hot-deck — df_hotdeck not in environment, skipping
[QA] Cluster-based — df_fastclus not in environment, skipping
[QA] SME — df_sme not in environment, skipping


In [12]:
# =========================================================
# QA ORCHESTRATOR — Rebuild-if-missing + Rich QA + Audit Pack
# Place this at the end of your notebook and run once.
# =========================================================
import numpy as np
import pandas as pd
from pathlib import Path

# Optional: for cluster-based rebuild
try:
    from sklearn.cluster import KMeans
    SKLEARN_OK = True
except Exception:
    SKLEARN_OK = False

rng = np.random.default_rng(42)

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def qa_means_like(frame: pd.DataFrame, columns: list[str], label: str):
    cols = [c for c in columns if c in frame.columns]
    if not cols:
        print(f"[QA] {label} — no numeric cols found in {columns}")
        return pd.DataFrame()
    out = []
    for c in cols:
        s = frame[c]
        out.append({
            "variable": c,
            "n": int(s.shape[0]),
            "nmiss": int(s.isna().sum()),
            "mean": float(s.mean(skipna=True)) if pd.api.types.is_numeric_dtype(s) else np.nan,
            "median": float(s.median(skipna=True)) if pd.api.types.is_numeric_dtype(s) else np.nan,
            "std": float(s.std(skipna=True)) if pd.api.types.is_numeric_dtype(s) else np.nan,
            "min": float(s.min(skipna=True)) if pd.api.types.is_numeric_dtype(s) else np.nan,
            "p50": float(s.median(skipna=True)) if pd.api.types.is_numeric_dtype(s) else np.nan,
            "max": float(s.max(skipna=True)) if pd.api.types.is_numeric_dtype(s) else np.nan,
        })
    df_out = pd.DataFrame(out)
    print(f"\n[QA] {label}")
    display(df_out)
    return df_out

def qa_imputation_summary_numeric(frame: pd.DataFrame, raw_col: str, imp_col: str) -> pd.DataFrame:
    raw, imp = frame[raw_col], frame[imp_col]
    n = int(frame.shape[0])
    n_miss_raw = int(raw.isna().sum())
    n_miss_imp = int(imp.isna().sum())
    n_imputed  = int(((raw.isna()) & (imp.notna())).sum())
    n_changed  = int(((raw.notna()) & (imp.notna()) & (raw != imp)).sum())
    out = pd.DataFrame([{
        "series_raw": raw_col, "series_imp": imp_col, "n": n,
        "n_miss_raw": n_miss_raw, "n_miss_imp": n_miss_imp,
        "n_imputed_from_missing": n_imputed,
        "n_overwritten_when_present": n_changed
    }])
    display(out)
    return out

def qa_imputation_summary_categorical(frame: pd.DataFrame, raw_col: str, sme_col: str) -> pd.DataFrame:
    raw, sme = frame[raw_col], frame[sme_col]
    n = int(frame.shape[0])
    n_miss_raw = int(raw.isna().sum())
    n_miss_sme = int(sme.isna().sum())
    n_imputed  = int(((raw.isna()) & (sme.notna())).sum())
    n_changed  = int(((raw.notna()) & (sme.notna()) & (raw != sme)).sum())
    out = pd.DataFrame([{
        "series_raw": raw_col, "series_sme": sme_col, "n": n,
        "n_miss_raw": n_miss_raw, "n_miss_sme": n_miss_sme,
        "n_imputed_from_missing": n_imputed,
        "n_overwritten_when_present": n_changed
    }])
    display(out)
    return out

def qa_show_heads(frame: pd.DataFrame, cols: list[str], label: str, n: int = 10):
    keep = [c for c in cols if c in frame.columns]
    if not keep:
        print(f"[HEAD] {label} — columns not found: {cols}")
        return pd.DataFrame()
    print(f"\n=== HEAD: {label} ===")
    out = frame[keep].head(n)
    display(out)
    return out

# ---------------------------------------------------------
# Rebuild missing imputation DataFrames if needed
# Requires `df` to exist. If not, load it here.
# ---------------------------------------------------------
if 'df' not in globals():
    print("[SETUP] Base DataFrame `df` not found. Please load your CSV earlier or specify a path here.")
    # Example:
    # df = pd.read_csv(r"C:\...\with_missing_ifrs9_pit_pd.csv")

# ---------- 1) HOT-DECK (credit_utilization) ----------
if 'df' in globals() and 'df_hotdeck' not in globals():
    df_hotdeck = df.copy()
    if 'credit_utilization' in df_hotdeck.columns:
        df_hotdeck['credit_utilization_raw'] = df_hotdeck['credit_utilization']
        strata_all = [c for c in ['employment_status','marital_status','region','stage'] if c in df_hotdeck.columns]
        # fallback: if no strata columns exist, use empty strata (global donors)
        if not strata_all:
            strata_all = []

        s = df_hotdeck['credit_utilization_raw'].copy()

        # groupwise donor fill
        if strata_all:
            def fill_group(g):
                donors = g['credit_utilization_raw'].dropna().values
                if donors.size == 0:
                    return g['credit_utilization_raw']
                miss_idx = g['credit_utilization_raw'].isna()
                if miss_idx.any():
                    g.loc[miss_idx, 'credit_utilization_raw'] = rng.choice(donors, size=int(miss_idx.sum()))
                return g['credit_utilization_raw']

            df_hotdeck['credit_utilization_imp'] = (
                df_hotdeck.groupby(strata_all, dropna=False, group_keys=False)
                          .apply(fill_group)
            )
        else:
            donors = s.dropna().values
            if donors.size:
                miss_idx = s.isna()
                s.loc[miss_idx] = rng.choice(donors, size=int(miss_idx.sum()))
            df_hotdeck['credit_utilization_imp'] = s

        print("[REBUILD] df_hotdeck created.")
    else:
        print("[REBUILD] df_hotdeck skipped — column 'credit_utilization' not in df.")

# ---------- 2) CLUSTER-BASED (internal_score) ----------
if 'df' in globals() and 'df_fastclus' not in globals():
    df_fastclus = df.copy()
    if SKLEARN_OK and all(c in df_fastclus.columns for c in ['internal_score','credit_utilization']):
        cluster_vars = ['internal_score','credit_utilization']
        target_col   = 'internal_score'
        k = 5

        # keep raw
        df_fastclus['internal_score_raw'] = df_fastclus['internal_score']

        complete = df_fastclus.dropna(subset=cluster_vars).copy()
        if complete.empty:
            print("[REBUILD] df_fastclus — no complete rows for clustering; skipping.")
        else:
            km = KMeans(n_clusters=k, n_init='auto', random_state=42)
            complete['cluster'] = km.fit_predict(complete[cluster_vars])

            df_fastclus['cluster'] = np.nan
            mask_can = df_fastclus[cluster_vars].notna().all(axis=1)
            df_fastclus.loc[mask_can, 'cluster'] = km.predict(df_fastclus.loc[mask_can, cluster_vars])

            clus_means = complete.groupby('cluster')[target_col].mean().rename('target_stat')
            df_fastclus = df_fastclus.merge(clus_means, how='left', left_on='cluster', right_index=True)

            df_fastclus['internal_score_imp'] = df_fastclus[target_col].where(
                df_fastclus[target_col].notna(),
                df_fastclus['target_stat']
            )

            # Backfill any remaining NaNs (e.g., cluster NaN) with global mean
            if df_fastclus['internal_score_imp'].isna().any():
                global_mean = float(df_fastclus[target_col].mean(skipna=True))
                df_fastclus['internal_score_imp'] = df_fastclus['internal_score_imp'].fillna(global_mean)

            before = int(df_fastclus[target_col].isna().sum())
            after  = int(df_fastclus['internal_score_imp'].isna().sum())
            print(f"[REBUILD] df_fastclus created. [CLUSTER QA] {target_col}: nmiss before={before}, after={after}")

            df_fastclus.drop(columns=['target_stat'], inplace=True, errors='ignore')
    else:
        print("[REBUILD] df_fastclus skipped — scikit-learn missing or required columns not present.")

# ---------- 3) SME RULES ----------
if 'df' in globals() and 'df_sme' not in globals():
    df_sme = df.copy()
    # Employment_status_sme
    if 'employment_status' in df_sme.columns:
        df_sme['employment_status_sme'] = df_sme['employment_status']
        if 'age' in df_sme.columns:
            df_sme.loc[df_sme['employment_status_sme'].isna() & (df_sme['age'] < 25), 'employment_status_sme'] = 'Probationary'
        if all(c in df_sme.columns for c in ['dpd','internal_score']):
            df_sme.loc[
                df_sme['employment_status_sme'].isna() & (df_sme['dpd'] > 120) & (df_sme['internal_score'] < 450),
                'employment_status_sme'
            ] = 'Unverified'
    # Marital_status_sme
    if 'marital_status' in df_sme.columns and 'dependents' in df_sme.columns:
        df_sme['marital_status_sme'] = df_sme['marital_status']
        df_sme.loc[
            df_sme['marital_status_sme'].isna() & (df_sme['dependents'] >= 2),
            'marital_status_sme'
        ] = 'Married-assumed'
    print("[REBUILD] df_sme created.")

# ---------------------------------------------------------
# RUN QA — Each block prints + displays tables
# ---------------------------------------------------------
# Mode step (already printed earlier in your run)
print("\n[QA] Mode step (categoricals shown above) — OK")

# Hot-deck QA
if 'df_hotdeck' in globals() and 'credit_utilization_imp' in df_hotdeck.columns:
    qa_means_like(df_hotdeck, ["credit_utilization_raw","credit_utilization_imp"], "Hot-deck credit_utilization")
    qa_imputation_summary_numeric(df_hotdeck, "credit_utilization_raw","credit_utilization_imp")
    qa_show_heads(df_hotdeck, ["credit_utilization_raw","credit_utilization_imp"], "CREDIT_UTILIZATION (RAW vs IMP)", n=10)
else:
    print("[QA] Hot-deck — still unavailable (run or check columns).")

# Cluster-based QA
if 'df_fastclus' in globals() and 'internal_score_imp' in df_fastclus.columns:
    qa_means_like(df_fastclus, ["internal_score","internal_score_imp"], "Cluster-based internal_score")
    qa_imputation_summary_numeric(df_fastclus, "internal_score","internal_score_imp")
    # cluster distribution of imputations
    if 'cluster' in df_fastclus.columns:
        filled = df_fastclus[df_fastclus['internal_score'].isna() & df_fastclus['internal_score_imp'].notna()]
        if not filled.empty:
            dist = filled['cluster'].value_counts(dropna=False).rename_axis('cluster').to_frame('n_imputed')
            print("\n=== IMPUTED COUNT BY CLUSTER ===")
            display(dist)
    qa_show_heads(df_fastclus, ["internal_score","internal_score_imp","cluster"], "INTERNAL_SCORE (RAW vs IMP) + CLUSTER", n=12)
else:
    print("[QA] Cluster-based — still unavailable (run or check columns / sklearn).")

# SME QA
if 'df_sme' in globals():
    cat_pairs = []
    if all(c in df_sme.columns for c in ["employment_status","employment_status_sme"]):
        cat_pairs.append(("employment_status","employment_status_sme"))
    if all(c in df_sme.columns for c in ["marital_status","marital_status_sme"]):
        cat_pairs.append(("marital_status","marital_status_sme"))

    for raw_col, sme_col in cat_pairs:
        qa_show_heads(df_sme, [raw_col, sme_col], f"{raw_col.upper()} (RAW vs SME)", n=8)
        # top counts
        vc = df_sme[sme_col].value_counts(dropna=False).reset_index()
        vc.columns = [sme_col, "count"]
        vc["pct"] = (vc["count"] / vc["count"].sum() * 100).round(2)
        print(f"\n=== SME QA: {sme_col.upper()} — TOP COUNTS (incl. NaN) ===")
        display(vc.head(12))
        print(f"\n=== SME QA: {raw_col.upper()} vs {sme_col.upper()} — SUMMARY ===")
        qa_imputation_summary_categorical(df_sme, raw_col, sme_col)

        # show where SME filled from missing raw for lecture illustration
        filled = df_sme[df_sme[raw_col].isna() & df_sme[sme_col].notna()].copy()
        cols_to_show = [c for c in [globals().get('id_var', None), raw_col, sme_col] if c]
        print(f"\n=== SME FILLS (RAW missing → SME provided) — sample {min(20,len(filled))} rows ===")
        display(filled[cols_to_show].head(20))
else:
    print("[QA] SME — still unavailable (run or check columns).")

# ---------------------------------------------------------
# Optional: Export an Audit Pack for your Udemy lecture
# ---------------------------------------------------------
OUT_XLSX = Path("IFRS9_Imputation_QA_AuditPack.xlsx")
with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as xl:
    # Hot-deck
    if 'df_hotdeck' in globals() and 'credit_utilization_imp' in df_hotdeck.columns:
        df_hotdeck[[c for c in [globals().get('id_var','account_id'),
                                'credit_utilization_raw','credit_utilization_imp'] if c in df_hotdeck.columns]]\
            .head(200).to_excel(xl, sheet_name="hotdeck_HEAD", index=False)
        qa_means_like(df_hotdeck, ["credit_utilization_raw","credit_utilization_imp"], "Hot-deck export")
        # re-run summary to capture to a sheet
        hd_sum = qa_imputation_summary_numeric(df_hotdeck, "credit_utilization_raw","credit_utilization_imp")
        hd_sum.to_excel(xl, sheet_name="hotdeck_SUMMARY", index=False)

    # Cluster-based
    if 'df_fastclus' in globals() and 'internal_score_imp' in df_fastclus.columns:
        cols = [c for c in [globals().get('id_var','account_id'),
                            'internal_score','internal_score_imp','cluster','credit_utilization'] if c in df_fastclus.columns]
        df_fastclus[cols].head(200).to_excel(xl, sheet_name="fastclus_HEAD", index=False)
        cl_sum = qa_imputation_summary_numeric(df_fastclus, "internal_score","internal_score_imp")
        cl_sum.to_excel(xl, sheet_name="fastclus_SUMMARY", index=False)
        # cluster imputed distribution
        if 'cluster' in df_fastclus.columns:
            filled = df_fastclus[df_fastclus['internal_score'].isna() & df_fastclus['internal_score_imp'].notna()]
            if not filled.empty:
                dist = filled['cluster'].value_counts(dropna=False).rename_axis('cluster').to_frame('n_imputed').reset_index()
                dist.to_excel(xl, sheet_name="fastclus_IMPUTED_BY_CLUSTER", index=False)

    # SME
    if 'df_sme' in globals():
        if all(c in df_sme.columns for c in ['employment_status','employment_status_sme']):
            df_sme[[c for c in [globals().get('id_var','account_id'),'employment_status','employment_status_sme'] if c in df_sme.columns]]\
                .head(200).to_excel(xl, sheet_name="sme_emp_HEAD", index=False)
            smes = qa_imputation_summary_categorical(df_sme, "employment_status","employment_status_sme")
            smes.to_excel(xl, sheet_name="sme_emp_SUMMARY", index=False)
        if all(c in df_sme.columns for c in ['marital_status','marital_status_sme']):
            df_sme[[c for c in [globals().get('id_var','account_id'),'marital_status','marital_status_sme'] if c in df_sme.columns]]\
                .head(200).to_excel(xl, sheet_name="sme_mar_HEAD", index=False)
            smms = qa_imputation_summary_categorical(df_sme, "marital_status","marital_status_sme")
            smms.to_excel(xl, sheet_name="sme_mar_SUMMARY", index=False)

print(f"\n[AUDIT PACK] Saved: {OUT_XLSX.resolve()}")
print("Use these printed tables + the Excel to narrate the QA section in your Udemy lecture.")


C:\Users\customer\AppData\Local\Temp\ipykernel_13768\2611614368.py:121: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(fill_group)


[REBUILD] df_hotdeck created.
[REBUILD] df_fastclus created. [CLUSTER QA] internal_score: nmiss before=500, after=0
[REBUILD] df_sme created.

[QA] Mode step (categoricals shown above) — OK

[QA] Hot-deck credit_utilization


,variable,n,nmiss,mean,median,std,min,p50,max
0,credit_utilization_raw,5000,500,0.756007,0.76,0.432513,0.0,0.76,1.5
1,credit_utilization_imp,5000,3,0.758855,0.77,0.432379,0.0,0.77,1.5


,series_raw,series_imp,n,n_miss_raw,n_miss_imp,n_imputed_from_missing,n_overwritten_when_present
0,credit_utilization_raw,credit_utilization_imp,5000,500,3,497,0



=== HEAD: CREDIT_UTILIZATION (RAW vs IMP) ===


...

,series_raw,series_imp,n,n_miss_raw,n_miss_imp,n_imputed_from_missing,n_overwritten_when_present
0,credit_utilization_raw,credit_utilization_imp,5000,500,3,497,0


,series_raw,series_imp,n,n_miss_raw,n_miss_imp,n_imputed_from_missing,n_overwritten_when_present
0,internal_score,internal_score_imp,5000,500,0,500,0


,series_raw,series_sme,n,n_miss_raw,n_miss_sme,n_imputed_from_missing,n_overwritten_when_present
0,employment_status,employment_status_sme,5000,500,423,77,0


,series_raw,series_sme,n,n_miss_raw,n_miss_sme,n_imputed_from_missing,n_overwritten_when_present
0,marital_status,marital_status_sme,5000,500,219,281,0



[AUDIT PACK] Saved: C:\Users\customer\Desktop\MyVideos\UDEMY LESSONS\IFRS 9 Python\DQ\Imputation\IFRS9_Imputation_QA_AuditPack.xlsx
Use these printed tables + the Excel to narrate the QA section in your Udemy lecture.
